In [1]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pathlib import Path
from datetime import date, timedelta
import math
import polars as pl

PROJECT_PATH = Path("/content/drive/MyDrive/multimodal-fashion-recsys")
PROCESSED_PATH = PROJECT_PATH / "data" / "processed"

VAL_START = date(2020, 9, 9)
K = 12

In [3]:
train = pl.scan_parquet(PROCESSED_PATH / "train.parquet")
validation_ground_truth = pl.read_parquet(PROCESSED_PATH / "validation_ground_truth.parquet")
article_mapping = pl.read_parquet(PROCESSED_PATH / "article_mapping.parquet")

print("Train:", train.select(pl.len()).collect().item())
print("Validation users:", validation_ground_truth.height)
print("Catalog size:", article_mapping.height)

Train: 31292772
Validation users: 72019
Catalog size: 105542


In [4]:
def average_precision_at_k(actual, predicted, k=12):
    actual = set(actual)
    if not actual:
        return 0.0

    score = 0.0
    hits = 0
    seen = set()

    for i, item in enumerate(predicted[:k], 1):
        if item in actual and item not in seen:
            hits += 1
            score += hits / i
        seen.add(item)

    return score / min(len(actual), k)


def recall_at_k(actual, predicted, k=12):
    actual = set(actual)
    if not actual:
        return 0.0

    return len(actual.intersection(predicted[:k])) / len(actual)


def ndcg_at_k(actual, predicted, k=12):
    actual = set(actual)
    if not actual:
        return 0.0

    dcg = sum(1 / math.log2(i + 2) for i, item in enumerate(predicted[:k]) if item in actual)
    idcg = sum(1 / math.log2(i + 2) for i in range(min(len(actual), k)))

    return dcg / idcg

In [5]:
def evaluate_static(actuals, predicted, catalog_size, k=12):
    return {
        "MAP@12": sum(average_precision_at_k(actual, predicted, k) for actual in actuals) / len(actuals),
        "Recall@12": sum(recall_at_k(actual, predicted, k) for actual in actuals) / len(actuals),
        "NDCG@12": sum(ndcg_at_k(actual, predicted, k) for actual in actuals) / len(actuals),
        "Coverage": len(set(predicted[:k])) / catalog_size
    }

In [6]:
actuals = validation_ground_truth["actual"].to_list()
catalog_size = article_mapping.height

## Popularity Baselines

In [7]:
global_top12 = (
    train
    .group_by("article_idx")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
    .head(K)
    .collect()["article_idx"]
    .to_list()
)

global_top12

[53893, 53894, 1714, 24838, 70222, 3712, 1715, 24837, 2237, 58492, 68, 53895]

In [8]:
global_metrics = evaluate_static(actuals, global_top12, catalog_size)

global_metrics

{'MAP@12': 0.0034440087457057516,
 'Recall@12': 0.008970650289728576,
 'NDCG@12': 0.006318058900555215,
 'Coverage': 0.00011369881184741619}

In [9]:
results = [{
    "model": "Global popularity",
    **global_metrics
}]

for days in [7, 14, 28, 56]:
    start_date = VAL_START - timedelta(days=days)

    top12 = (
        train
        .filter(pl.col("t_dat") >= start_date)
        .group_by("article_idx")
        .agg(pl.len().alias("count"))
        .sort("count", descending=True)
        .head(K)
        .collect()["article_idx"]
        .to_list()
    )

    results.append({
        "model": f"Recent popularity {days}d",
        **evaluate_static(actuals, top12, catalog_size)
    })

In [10]:
results = pl.DataFrame(results).sort("MAP@12", descending=True)

results

model,MAP@12,Recall@12,NDCG@12,Coverage
str,f64,f64,f64,f64
"""Recent popularity 14d""",0.006995,0.022269,0.013167,0.000114
"""Recent popularity 7d""",0.006604,0.020905,0.012605,0.000114
"""Recent popularity 28d""",0.005593,0.021016,0.011817,0.000114
"""Recent popularity 56d""",0.004206,0.012859,0.008406,0.000114
"""Global popularity""",0.003444,0.008971,0.006318,0.000114


## Personal History + Recent Popularity

In [11]:
BEST_DAYS = 14

recent_top100 = (
    train
    .filter(pl.col("t_dat") >= VAL_START - timedelta(days=BEST_DAYS))
    .group_by("article_idx")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
    .head(100)
    .collect()["article_idx"]
    .to_list()
)

recent_top100[:12]

[103794,
 67523,
 67544,
 101719,
 53893,
 103797,
 104046,
 94675,
 105147,
 103795,
 103796,
 101368]

In [12]:
val_users = validation_ground_truth.select("customer_idx")

user_history = (
    train
    .join(val_users.lazy(), on="customer_idx", how="semi")
    .sort(["customer_idx", "t_dat"], descending=[False, True])
    .group_by("customer_idx", maintain_order=True)
    .agg(pl.col("article_idx").unique(maintain_order=True).head(K).alias("history"))
    .collect()
)

user_history.head()

customer_idx,history
u32,list[u32]
3,"[92160, 92159, … 40180]"
7,"[75962, 57069, … 62289]"
39,"[105274, 100028, … 17132]"
87,"[102711, 103584, … 96636]"
91,"[87649, 93821, … 58374]"


In [13]:
def combine_recommendations(history, fallback, k=12):
    recommendations = []
    seen = set()

    for item in (history or []) + fallback:
        if item not in seen:
            recommendations.append(item)
            seen.add(item)

        if len(recommendations) == k:
            break

    return recommendations

In [14]:
evaluation = validation_ground_truth.join(user_history, on="customer_idx", how="left")

predictions = [
    combine_recommendations(history, recent_top100, K)
    for history in evaluation["history"].to_list()
]

actuals_personal = evaluation["actual"].to_list()

In [15]:
def evaluate_personalized(actuals, predictions, catalog_size, k=12):
    return {
        "MAP@12": sum(average_precision_at_k(actual, predicted, k) for actual, predicted in zip(actuals, predictions)) / len(actuals),
        "Recall@12": sum(recall_at_k(actual, predicted, k) for actual, predicted in zip(actuals, predictions)) / len(actuals),
        "NDCG@12": sum(ndcg_at_k(actual, predicted, k) for actual, predicted in zip(actuals, predictions)) / len(actuals),
        "Coverage": len({item for predicted in predictions for item in predicted[:k]}) / catalog_size
    }

In [16]:
personal_metrics = evaluate_personalized(actuals_personal, predictions, catalog_size)

personal_metrics

{'MAP@12': 0.02436360074510926,
 'Recall@12': 0.04435239698057186,
 'NDCG@12': 0.03369904832340075,
 'Coverage': 0.4877300032214663}

In [17]:
results = pl.concat([
    results,
    pl.DataFrame([{
        "model": "Personal history + recent popularity",
        **personal_metrics
    }])
], how="vertical_relaxed").sort("MAP@12", descending=True)

results

model,MAP@12,Recall@12,NDCG@12,Coverage
str,f64,f64,f64,f64
"""Personal history + recent popu…",0.024364,0.044352,0.033699,0.48773
"""Recent popularity 14d""",0.006995,0.022269,0.013167,0.000114
"""Recent popularity 7d""",0.006604,0.020905,0.012605,0.000114
"""Recent popularity 28d""",0.005593,0.021016,0.011817,0.000114
"""Recent popularity 56d""",0.004206,0.012859,0.008406,0.000114
"""Global popularity""",0.003444,0.008971,0.006318,0.000114


### History Window Comparison

In [18]:
personal_results = [{
    "model": "All history",
    **personal_metrics
}]

for days in [14, 28, 56, 112]:
    start_date = VAL_START - timedelta(days=days)

    history = (
        train
        .filter(pl.col("t_dat") >= start_date)
        .join(val_users.lazy(), on="customer_idx", how="semi")
        .sort(["customer_idx", "t_dat"], descending=[False, True])
        .group_by("customer_idx", maintain_order=True)
        .agg(pl.col("article_idx").unique(maintain_order=True).head(K).alias("history"))
        .collect()
    )

    evaluation = validation_ground_truth.join(history, on="customer_idx", how="left")

    predictions = [
        combine_recommendations(history, recent_top100, K)
        for history in evaluation["history"].to_list()
    ]

    personal_results.append({
        "model": f"Last {days}d history",
        **evaluate_personalized(evaluation["actual"].to_list(), predictions, catalog_size)
    })

In [19]:
personal_results = pl.DataFrame(personal_results).sort("MAP@12", descending=True)

personal_results

model,MAP@12,Recall@12,NDCG@12,Coverage
str,f64,f64,f64,f64
"""Last 56d history""",0.02517,0.049558,0.036092,0.218965
"""Last 28d history""",0.024928,0.049248,0.035886,0.173362
"""Last 112d history""",0.024904,0.047826,0.035262,0.270357
"""All history""",0.024364,0.044352,0.033699,0.48773
"""Last 14d history""",0.023758,0.0467,0.034205,0.133814


In [20]:
best_personal = personal_results.row(0, named=True)

baseline_results = pl.concat([
    results.filter(pl.col("model") != "Personal history + recent popularity"),
    pl.DataFrame([{
        "model": "Personal history 56d + recent popularity 14d",
        "MAP@12": best_personal["MAP@12"],
        "Recall@12": best_personal["Recall@12"],
        "NDCG@12": best_personal["NDCG@12"],
        "Coverage": best_personal["Coverage"]
    }])
], how="vertical_relaxed").sort("MAP@12", descending=True)

baseline_results

model,MAP@12,Recall@12,NDCG@12,Coverage
str,f64,f64,f64,f64
"""Personal history 56d + recent …",0.02517,0.049558,0.036092,0.218965
"""Recent popularity 14d""",0.006995,0.022269,0.013167,0.000114
"""Recent popularity 7d""",0.006604,0.020905,0.012605,0.000114
"""Recent popularity 28d""",0.005593,0.021016,0.011817,0.000114
"""Recent popularity 56d""",0.004206,0.012859,0.008406,0.000114
"""Global popularity""",0.003444,0.008971,0.006318,0.000114
